In [1]:
import pandas as pd

df = pd.read_csv('downloads/netflix_customer_churn.csv')
df.head()
import sqlite3

conn = sqlite3.connect('netflix_churn.db')
df.to_sql('customers', conn, if_exists='replace', index=False)

5000

In [2]:
df.duplicated().sum()
df['customer_id'].duplicated().sum()
for col in ['subscription_type', 'region', 'device', 'payment_method', 'favorite_genre']:
    print(df[col].value_counts())
    print()
df['churned'].value_counts(normalize=True) * 100    

subscription_type
Premium     1693
Basic       1661
Standard    1646
Name: count, dtype: int64

region
South America    873
Europe           867
North America    851
Asia             841
Africa           803
Oceania          765
Name: count, dtype: int64

device
Tablet     1048
Laptop     1006
Mobile     1004
TV          993
Desktop     949
Name: count, dtype: int64

payment_method
Debit Card     1030
PayPal         1026
Crypto          995
Gift Card       976
Credit Card     973
Name: count, dtype: int64

favorite_genre
Drama          731
Documentary    729
Romance        725
Sci-Fi         720
Horror         713
Action         697
Comedy         685
Name: count, dtype: int64



churned
1    50.3
0    49.7
Name: proportion, dtype: float64

In [3]:
df.groupby('subscription_type')['churned'].mean().mul(100).round(1)

subscription_type
Basic       61.8
Premium     43.7
Standard    45.4
Name: churned, dtype: float64

In [4]:
df.groupby('region')['churned'].mean().mul(100).round(1)

region
Africa           48.3
Asia             50.7
Europe           51.7
North America    49.5
Oceania          50.1
South America    51.4
Name: churned, dtype: float64

In [5]:
df.groupby('favorite_genre')['churned'].mean().mul(100).round(1)

favorite_genre
Action         52.4
Comedy         49.9
Documentary    50.8
Drama          52.3
Horror         51.5
Romance        48.3
Sci-Fi         47.1
Name: churned, dtype: float64

In [6]:
df.groupby('churned')[['watch_hours', 'avg_watch_time_per_day', 'last_login_days', 'monthly_fee', 'number_of_profiles', 'age']].mean().round(2)

,watch_hours,avg_watch_time_per_day,last_login_days,monthly_fee,number_of_profiles,age
churned,,,,,,
0,17.45,1.59,21.77,14.25,3.25,43.90
1,5.92,0.16,38.31,13.13,2.80,43.79


In [7]:
df['login_recency_bucket'] = pd.cut(
    df['last_login_days'],
    bins=[-1, 7, 14, 30, df['last_login_days'].max()],
    labels=['0-7 days', '8-14 days', '15-30 days', '30+ days']
)

df.groupby('login_recency_bucket')['churned'].mean().mul(100).round(1)

/var/folders/1b/kw0vjyj96sn_xmt6t7qmymj80000gn/T/ipykernel_16721/3160936391.py:7: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby('login_recency_bucket')['churned'].mean().mul(100).round(1)


login_recency_bucket
0-7 days      13.4
8-14 days     27.0
15-30 days    32.0
30+ days      75.1
Name: churned, dtype: float64

In [8]:
churned_customers = df['churned'].sum()
monthly_revenue_lost = df[df['churned'] == 1]['monthly_fee'].sum()

print(f"Churned customers: {churned_customers}")
print(f"Monthly revenue lost: ${monthly_revenue_lost:,.2f}")

Churned customers: 2515
Monthly revenue lost: $33,009.85


In [9]:
df[df['churned'] == 1].groupby('subscription_type')['monthly_fee'].sum().round(2)

subscription_type
Basic        9232.73
Premium     13312.60
Standard    10464.52
Name: monthly_fee, dtype: float64

In [10]:
total_monthly_revenue = df['monthly_fee'].sum()
pct_revenue_lost = (monthly_revenue_lost / total_monthly_revenue) * 100
print(f"Total monthly revenue: ${total_monthly_revenue:,.2f}")
print(f"% of revenue at risk from churn: {pct_revenue_lost:.1f}%")

Total monthly revenue: $68,417.00
% of revenue at risk from churn: 48.2%


In [11]:
df.to_csv('netflix_churn_cleaned.csv', index=False)

In [12]:
query = """
SELECT subscription_type,
       COUNT(*) AS total_customers,
       SUM(churned) AS churned_customers,
       ROUND(100.0 * SUM(churned) / COUNT(*), 1) AS churn_rate_pct
FROM customers
GROUP BY subscription_type
ORDER BY churn_rate_pct DESC;
"""
pd.read_sql_query(query, conn)

,subscription_type,total_customers,churned_customers,churn_rate_pct
0,Basic,1661,1027,61.8
1,Standard,1646,748,45.4
2,Premium,1693,740,43.7


In [13]:
query2 = """
SELECT 
    ROUND(SUM(monthly_fee), 2) AS total_monthly_revenue,
    ROUND(SUM(CASE WHEN churned = 1 THEN monthly_fee ELSE 0 END), 2) AS revenue_at_risk,
    ROUND(100.0 * SUM(CASE WHEN churned = 1 THEN monthly_fee ELSE 0 END) / SUM(monthly_fee), 1) AS pct_revenue_at_risk
FROM customers;
"""
pd.read_sql_query(query2, conn)

,total_monthly_revenue,revenue_at_risk,pct_revenue_at_risk
0,68417.0,33009.85,48.2


In [14]:
query3 = """
SELECT 
  CASE 
    WHEN last_login_days <= 7 THEN '0-7 days'
    WHEN last_login_days <= 14 THEN '8-14 days'
    WHEN last_login_days <= 30 THEN '15-30 days'
    ELSE '30+ days'
  END AS login_recency_bucket,
  COUNT(*) AS total_customers,
  SUM(churned) AS churned_customers,
  ROUND(100.0 * SUM(churned) / COUNT(*), 1) AS churn_rate_pct
FROM customers
GROUP BY login_recency_bucket
ORDER BY MIN(last_login_days);
"""
pd.read_sql_query(query3, conn)

,login_recency_bucket,total_customers,churned_customers,churn_rate_pct
0,0-7 days,650,87,13.4
1,8-14 days,559,151,27.0
2,15-30 days,1322,423,32.0
3,30+ days,2469,1854,75.1
